In [1]:
import sys
sys.path.append("..")
import glob, json
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score
from src.preprocess import preprocess_image

tf.keras.utils.set_random_seed(42)

with open("../data/idx_to_letter.json") as f:
    idx_to_letter = {int(k): v for k, v in json.load(f).items()}
letter_to_idx = {v: k for k, v in idx_to_letter.items()}

paths = []
for ext in ("jpg", "jpeg", "JPG", "JPEG", "png", "PNG"):
    paths += glob.glob(f"../data/real/*/*/*.{ext}")
paths = sorted(set(paths))

rows, X = [], []
for path in paths:
    parts = path.replace("\\", "/").split("/")
    person, letter = parts[-3], parts[-2].upper()
    if letter not in letter_to_idx:
        continue
    img = cv2.imread(path)
    if img is None:
        continue
    X.append(preprocess_image(img))
    rows.append({"person": person, "letter": letter,
                 "y": letter_to_idx[letter], "path": path})

X = np.array(X)
df = pd.DataFrame(rows)
y = df["y"].values
print(df.groupby("person").size())

person
malak    110
sara      18
zineb     24
dtype: int64


In [2]:
train_persons = ["malak"]
test_persons = ["sara", "zineb"]

tr = df["person"].isin(train_persons).values
te = df["person"].isin(test_persons).values

X_real_tr, y_real_tr = X[tr], y[tr]
X_real_te, y_real_te = X[te], y[te]
print("Entraînement :", len(X_real_tr), "photos | Test :", len(X_real_te), "photos")

Entraînement : 110 photos | Test : 42 photos


In [3]:
data = np.load("../data/sign_mnist_prepared.npz")
X_mnist, y_mnist = data["X_train"], data["y_train"]

augment = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom((-0.2, 0.2), (-0.2, 0.2)),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.3, value_range=(0, 1)),
    layers.RandomContrast(0.3),
])

def apply_aug(x, y_):
    return tf.clip_by_value(augment(x, training=True), 0.0, 1.0), y_

In [4]:
REPS, N_MNIST, EPOCHS, LR = 20, 5000, 5, 1e-4

model = keras.models.load_model("../data/cnn_final.keras")
model.compile(optimizer=keras.optimizers.Adam(LR),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

rng = np.random.default_rng(0)
idx = rng.choice(len(X_mnist), N_MNIST, replace=False)
X_fit = np.concatenate([np.repeat(X_real_tr, REPS, axis=0), X_mnist[idx]])
y_fit = np.concatenate([np.repeat(y_real_tr, REPS), y_mnist[idx]])

ds = (tf.data.Dataset.from_tensor_slices((X_fit, y_fit))
      .shuffle(len(X_fit), seed=0).batch(64)
      .map(apply_aug, num_parallel_calls=tf.data.AUTOTUNE)
      .prefetch(tf.data.AUTOTUNE))

# Avant
base = keras.models.load_model("../data/cnn_final.keras")
pred_before = base.predict(X_real_te, verbose=0).argmax(1)

model.fit(ds, epochs=EPOCHS, verbose=1)
pred_after = model.predict(X_real_te, verbose=0).argmax(1)

print("\nTest sur Sara + Zineb (jamais vues à l'entraînement)")
print(f"Avant fine-tuning : {(pred_before == y_real_te).mean():.3f}")
print(f"Après fine-tuning : {(pred_after == y_real_te).mean():.3f}")
print(f"F1 macro avant : {f1_score(y_real_te, pred_before, average='macro'):.3f}")
print(f"F1 macro après : {f1_score(y_real_te, pred_after, average='macro'):.3f}")

Epoch 1/5


c:\Users\Malak\OneDrive\Documents\S2_EMSI\DM\projet_sign_language\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


113/113 ━━━━━━━━━━━━━━━━━━━━ 18s 114ms/step - accuracy: 0.7607 - loss: 1.2451
Epoch 2/5
113/113 ━━━━━━━━━━━━━━━━━━━━ 21s 182ms/step - accuracy: 0.7907 - loss: 0.8503
Epoch 3/5
113/113 ━━━━━━━━━━━━━━━━━━━━ 27s 240ms/step - accuracy: 0.8144 - loss: 0.6964
Epoch 4/5
113/113 ━━━━━━━━━━━━━━━━━━━━ 27s 232ms/step - accuracy: 0.8329 - loss: 0.5782
Epoch 5/5
113/113 ━━━━━━━━━━━━━━━━━━━━ 39s 218ms/step - accuracy: 0.8407 - loss: 0.5146

Test sur Sara + Zineb (jamais vues à l'entraînement)
Avant fine-tuning : 0.119
Après fine-tuning : 0.119
F1 macro avant : 0.037
F1 macro après : 0.088


In [5]:
df_te = df[te].copy()
df_te["ok_avant"] = pred_before == y_real_te
df_te["ok_apres"] = pred_after == y_real_te
print(df_te.groupby("person")[["ok_avant", "ok_apres"]].agg(["mean", "count"]).round(3))

# Le modèle a-t-il oublié MNIST ?
Xm_te, ym_te = data["X_test"], data["y_test"]
acc_mnist = (model.predict(Xm_te, verbose=0).argmax(1) == ym_te).mean()
print(f"\nAccuracy MNIST test après fine-tuning : {acc_mnist:.3f}")

# Lettre prédite à tort le plus souvent
wrong = pd.Series(pred_after[pred_after != y_real_te]).map(idx_to_letter)
print("\nLettres prédites à tort :\n", wrong.value_counts().head(5))

       ok_avant       ok_apres      
           mean count     mean count
person                              
sara      0.167    18    0.111    18
zineb     0.083    24    0.125    24

Accuracy MNIST test après fine-tuning : 0.997

Lettres prédites à tort :
 P    11
Q     6
O     6
H     5
C     3
Name: count, dtype: int64


In [6]:
model.save("../data/cnn_finetune_malak.keras")